# Analyze ONNX for GPT

Analyze one NeuroGolf ONNX file and generate a compact text package that can be pasted into GPT for optimization ideas.

In [31]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from neurogolf_score import score_model_file
from neurogolf_onnx_analysis import (
    compare_score_rows,
    compact_model_markdown,
    gpt_analysis_prompt,
    run_model_on_examples,
    summarize_model,
    summarize_task,
)

DATA_DIR = repo_root / "neurogolf-2026"
WORK_DIR = repo_root / "outputs" / "gpt_onnx_analysis"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TASK_ID = "task001"
ONNX_PATH = repo_root / "solution" / "6406.18" / f"{TASK_ID}.onnx"
CANDIDATE_ONNX_DIR = repo_root / "outputs" / "gpt_workbench" / TASK_ID
CANDIDATE_ONNX_PATHS = sorted(CANDIDATE_ONNX_DIR.glob("*.onnx"))

TASK_ID, ONNX_PATH, CANDIDATE_ONNX_DIR, CANDIDATE_ONNX_PATHS


('task001',
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6406.18/task001.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task001'),
 [PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task001/task001_no_p3_direct_slice.onnx')])

## Task Summary

In [32]:
task_summary = summarize_task(TASK_ID, DATA_DIR)

print(f"task    : {task_summary['task']}")
print(f"train   : {task_summary['num_train']}")
print(f"test    : {task_summary['num_test']}")
print(f"arc-gen : {task_summary['num_arc_gen']}")
print(f"colors  : {task_summary['color_counts']}")

try:
    import pandas as pd

    display(pd.DataFrame(task_summary["examples"]))
except ImportError:
    task_summary["examples"]

task    : task001
train   : 5
test    : 1
arc-gen : 262
colors  : {0: 15578, 1: 824, 2: 1106, 3: 608, 4: 742, 5: 1054, 6: 1170, 7: 1336, 8: 812, 9: 890}


,subset,index,input_shape,output_shape,input_cells,output_cells
0,train,0,3x3,9x9,9,81
1,train,1,3x3,9x9,9,81
2,train,2,3x3,9x9,9,81
3,train,3,3x3,9x9,9,81
4,train,4,3x3,9x9,9,81
...,...,...,...,...,...,...
263,arc-gen,257,3x3,9x9,9,81
264,arc-gen,258,3x3,9x9,9,81
265,arc-gen,259,3x3,9x9,9,81
266,arc-gen,260,3x3,9x9,9,81


## Score

In [33]:
base_score_row = score_model_file(ONNX_PATH, DATA_DIR)
candidate_score_rows = []

for candidate_path in CANDIDATE_ONNX_PATHS:
    row = score_model_file(candidate_path, DATA_DIR)
    row["candidate_file"] = candidate_path.name
    row["candidate_path"] = str(candidate_path)
    row.update(compare_score_rows(base_score_row, row))
    candidate_score_rows.append(row)

# Keep this alias for the GPT prompt generation cells below.
score_row = base_score_row

print("Base/reference ONNX:")
display(base_score_row)

print(f"Candidate ONNX files: {len(candidate_score_rows)}")
try:
    import pandas as pd

    if candidate_score_rows:
        score_cols = [
            "candidate_file", "status", "score", "score_delta", "cost", "cost_delta",
            "memory", "memory_delta", "params", "params_delta", "filesize", "filesize_delta", "error"
        ]
        display(pd.DataFrame(candidate_score_rows)[score_cols].sort_values("score", ascending=False).reset_index(drop=True))
    else:
        print(f"No candidate ONNX files found in {CANDIDATE_ONNX_DIR}")
except ImportError:
    candidate_score_rows


Base/reference ONNX:


{'task': 'task001',
 'candidate': '6406.18',
 'file': '/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6406.18/task001.onnx',
 'filesize': 1172,
 'status': 'ok',
 'memory': 2666,
 'params': 24,
 'cost': 2690.0,
 'score': 17.102703527404117,
 'error': ''}

Candidate ONNX files: 1


,candidate_file,status,score,score_delta,cost,cost_delta,memory,memory_delta,params,params_delta,filesize,filesize_delta,error
0,task001_no_p3_direct_slice.onnx,ok,17.150676,0.047973,2564.0,-126.0,2540,-126,24,0,1312,140,


## Candidate Comparison

In [34]:
base_validation = run_model_on_examples(ONNX_PATH, TASK_ID, DATA_DIR)
candidate_validation_rows = []
candidate_validations = {}

for candidate_path in CANDIDATE_ONNX_PATHS:
    report = run_model_on_examples(candidate_path, TASK_ID, DATA_DIR)
    candidate_validations[candidate_path.name] = report
    counts = report.get("counts", {})
    candidate_validation_rows.append({
        "candidate_file": candidate_path.name,
        "validation_status": report.get("status"),
        "pass": counts.get("pass", 0),
        "fail": counts.get("fail", 0),
        "error": counts.get("error", 0),
        "skipped_large_grid": counts.get("skipped_large_grid", 0),
        "first_failure": report.get("first_failure"),
    })

print("Base validation:")
display({k: v for k, v in base_validation.items() if k != "rows"})

print("Candidate validations:")
try:
    import pandas as pd

    validation_df = pd.DataFrame(candidate_validation_rows)
    if candidate_score_rows:
        score_df = pd.DataFrame(candidate_score_rows)
        compare_df = validation_df.merge(
            score_df[["candidate_file", "status", "score", "score_delta", "cost_delta", "memory_delta", "params_delta", "filesize_delta"]],
            on="candidate_file",
            how="left",
        )
        display(compare_df.sort_values(["validation_status", "score_delta"], ascending=[True, False]).reset_index(drop=True))
    else:
        display(validation_df)
except ImportError:
    candidate_validation_rows


Base validation:


{'status': 'ok', 'counts': {'pass': 268}, 'first_failure': None}

Candidate validations:


,candidate_file,validation_status,pass,fail,error,skipped_large_grid,first_failure,status,score,score_delta,cost_delta,memory_delta,params_delta,filesize_delta
0,task001_no_p3_direct_slice.onnx,ok,268,0,0,0,None,ok,17.150676,0.047973,-126.0,-126,0,140


In [35]:
valid_improvements = []
for row in candidate_score_rows:
    validation = candidate_validations.get(row["candidate_file"], {})
    if validation.get("status") == "ok" and row.get("score_delta", 0) > 0:
        valid_improvements.append(row)

print(f"Valid score improvements: {len(valid_improvements)}")
try:
    import pandas as pd

    if valid_improvements:
        display(pd.DataFrame(valid_improvements).sort_values("score_delta", ascending=False).reset_index(drop=True))
except ImportError:
    valid_improvements


Valid score improvements: 1


,task,candidate,file,filesize,status,memory,params,cost,score,error,candidate_file,candidate_path,filesize_delta,memory_delta,params_delta,cost_delta,score_delta,base_status,candidate_status
0,task001_no_p3_direct_slice,task001,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,1312,ok,2540,24,2564.0,17.150676,,task001_no_p3_direct_slice.onnx,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,140,-126,0,-126.0,0.047973,ok,ok


In [36]:
failed_candidates = {
    name: report.get("first_failure")
    for name, report in candidate_validations.items()
    if report.get("first_failure")
}
failed_candidates


{}

## ONNX Structure

In [37]:
model_summary = summarize_model(ONNX_PATH)

print(f"file         : {model_summary['filename']}")
print(f"filesize     : {model_summary['filesize']}")
print(f"ir_version   : {model_summary['ir_version']}")
print(f"opset_imports: {model_summary['opset_imports']}")
print(f"inputs       : {model_summary['inputs']}")
print(f"outputs      : {model_summary['outputs']}")
print(f"nodes        : {model_summary['num_nodes']}")
print(f"initializers : {model_summary['num_initializers']}")
print(f"op_counts    : {model_summary['op_counts']}")

file         : task001.onnx
filesize     : 1172
ir_version   : 8
opset_imports: [{'domain': '', 'version': 9}]
inputs       : [{'name': 'input', 'elem_type': 'FLOAT', 'shape': [1, 10, 30, 30]}]
outputs      : [{'name': 'output', 'elem_type': 'FLOAT16', 'shape': [1, 10, 30, 30]}]
nodes        : 14
initializers : 4
op_counts    : {'Cast': 3, 'Concat': 1, 'Gather': 2, 'Or': 1, 'Pad': 1, 'ReduceMax': 1, 'Slice': 3, 'Tile': 1, 'Where': 1}


## Initializers

In [38]:
try:
    import pandas as pd

    display(pd.DataFrame(model_summary["initializers"]))
except ImportError:
    model_summary["initializers"]

,name,dtype,dims,numel,bytes,sample
0,tile_repeats_mask,INT64,[4],4,32,"[1, 1, 3, 3]"
1,blk_idx,INT64,[9],9,72,"[0, 0, 0, 1, 1, 1, 2, 2, 2]"
2,bg_onehot,FLOAT16,"[1, 10, 1, 1]",10,20,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,fg_bg_zero,FLOAT16,"[1, 1, 1, 1]",1,2,[0.0]


## Nodes

In [39]:
try:
    import pandas as pd

    nodes_df = pd.DataFrame(model_summary["nodes"])
    display(nodes_df[["index", "op_type", "name", "inputs", "outputs", "attributes"]])
except ImportError:
    model_summary["nodes"][:20]

,index,op_type,name,inputs,outputs,attributes
0,0,Slice,20,[input],[p3],"{'axes': [2, 3], 'ends': [3, 3], 'starts': [0,..."
1,1,Slice,21,[p3],[p3_bg],"{'axes': [1], 'ends': [1], 'starts': [0]}"
2,2,Cast,25,[p3_bg],[bg3b],{'to': 9}
3,3,Tile,26,"[bg3b, tile_repeats_mask]",[pat_bg9],{}
4,4,Gather,27,"[bg3b, blk_idx]",[sel_bg_rows],{'axis': 2}
5,5,Gather,28,"[sel_bg_rows, blk_idx]",[sel_bg9],{'axis': 3}
6,6,Or,29,"[pat_bg9, sel_bg9]",[out_bg_mask],{}
7,7,Cast,foldcast_3_0,[p3],[p3__f16_3_0],{'to': 2}
8,8,Slice,22,[p3__f16_3_0],[p3_fg__h16],"{'axes': [1], 'ends': [10], 'starts': [1]}"
9,9,Cast,p3_fg__h16_u8up,[p3_fg__h16],[p3_fg__h16_u8tof16],{'to': 10}


## GPT Package

In [40]:
model_markdown = compact_model_markdown(model_summary, score_row=score_row, max_nodes=160)
prompt = gpt_analysis_prompt(task_summary, model_markdown)

prompt_path = WORK_DIR / f"{TASK_ID}_gpt_analysis_prompt.md"
prompt_path.write_text(prompt)

print(f"Saved GPT prompt: {prompt_path}")
print(f"Prompt characters: {len(prompt)}")

Saved GPT prompt: /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_onnx_analysis/task001_gpt_analysis_prompt.md
Prompt characters: 35242


In [41]:
print(prompt)

You are helping optimize a NeuroGolf ONNX model.

Rules:
- Input tensor: float32 [1, 10, 30, 30], name 'input'.
- Output tensor: float32 [1, 10, 30, 30], name 'output'.
- Static tensor shapes only.
- One input and one output only.
- Banned ops: Loop, Scan, NonZero, Unique, Script, Function, Compress.
- Optimize score by reducing memory + params.
- Assume public examples pass unless told otherwise.

Requested output:
1. Summarize what the current ONNX appears to do.
2. Identify redundant or expensive parts.
3. Propose safe rewrite candidates ranked by risk and likely score gain.
4. Do not write code yet.

Task summary:
- task: task001
- train examples: 5
- test examples: 1
- arc-gen examples: 262
- color counts: {0: 15578, 1: 824, 2: 1106, 3: 608, 4: 742, 5: 1054, 6: 1170, 7: 1336, 8: 812, 9: 890}
- example shapes: [{'subset': 'train', 'index': 0, 'input_shape': '3x3', 'output_shape': '9x9', 'input_cells': 9, 'output_cells': 81}, {'subset': 'train', 'index': 1, 'input_shape': '3x3', 'ou

Suggested workflow: paste the generated prompt into GPT first and ask for analysis only. After choosing a candidate rewrite, ask GPT to produce a `build_taskXXX.py` script that creates a new ONNX.